<a href="https://colab.research.google.com/github/sanjananagendra16-cpu/SIH2026/blob/main/SIH_Crop_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import tensorflow_datasets as tfds

print(tfds)
print(tfds.__file__)
print(hasattr(tfds, "load"))

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/co

<module 'tensorflow_datasets' from '/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py'>
/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py
False


In [ ]:
!git clone -q https://github.com/spMohanty/PlantVillage-Dataset.git
print("PlantVillage downloaded!")

PlantVillage downloaded!


In [ ]:
import os

path = "/content/PlantVillage-Dataset/raw/color"

print(os.listdir(path)[:10])
print("Number of classes:", len(os.listdir(path)))

['Potato___healthy', 'Apple___Black_rot', 'Tomato___healthy', 'Soybean___healthy', 'Apple___healthy', 'Apple___Cedar_apple_rust', 'Pepper,_bell___healthy', 'Tomato___Tomato_mosaic_virus', 'Tomato___Target_Spot', 'Squash___Powdery_mildew']
Number of classes: 38


In [ ]:
import os
import shutil

source = "/content/PlantVillage-Dataset/raw/color"
target = "/content/potato_dataset"

classes = [
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy"
]

# Start fresh
if os.path.exists(target):
    shutil.rmtree(target)

os.makedirs(target)

# Use the same number from every class
IMAGES_PER_CLASS = 150

for cls in classes:
    src = os.path.join(source, cls)
    dst = os.path.join(target, cls)
    os.makedirs(dst)

    files = os.listdir(src)[:IMAGES_PER_CLASS]

    for file in files:
        shutil.copy(
            os.path.join(src, file),
            os.path.join(dst, file)
        )

print("Balanced dataset prepared!\n")

for cls in classes:
    print(cls, ":", len(os.listdir(os.path.join(target, cls))))

Balanced dataset prepared!

Potato___Early_blight : 150
Potato___Late_blight : 150
Potato___healthy : 150


In [ ]:
import tensorflow as tf

IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/potato_dataset",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/potato_dataset",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("Classes:", class_names)

Found 450 files belonging to 3 classes.
Using 360 files for training.
Found 450 files belonging to 3 classes.
Using 90 files for validation.
Classes: ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step - accuracy: 0.3861 - loss: 1.4157 - val_accuracy: 0.7111 - val_loss: 0.7179
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.7694 - loss: 0.5725 - val_accuracy: 0.8667 - val_loss: 0.3583
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8944 - loss: 0.3342 - val_accuracy: 0.9444 - val_loss: 0.2356
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9333 - loss: 0.2328 - val_accuracy: 0.9444 - val_loss: 0.1944
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9444 - loss: 0.1999 - val_accuracy: 0.9444 - val_loss: 0.1861


In [ ]:
import numpy as np
import tensorflow as tf

class_names = [
    "Potato Early Blight",
    "Potato Late Blight",
    "Potato Healthy"
]

def predict_disease(image):
    # Resize image
    img = tf.image.resize(image, (160, 160))

    # Convert to array
    img = tf.cast(img, tf.float32) / 255.0

    # Add batch dimension
    img = tf.expand_dims(img, 0)

    # Prediction
    predictions = model.predict(img, verbose=0)[0]

    index = np.argmax(predictions)
    confidence = predictions[index] * 100

    disease = class_names[index]

    return disease, confidence

In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()

for filename in uploaded.keys():
    image = Image.open(filename).convert("RGB")
    image = np.array(image)

    disease, confidence = predict_disease(image)

    print("🌱 CROP DISEASE DETECTION")
    print("-------------------------")
    print("Prediction:", disease)
    print(f"Confidence: {confidence:.2f}%")

Saving Screenshot_18-9-2026_223456_bpb-us-e1.wpmucdn.com.jpeg to Screenshot_18-9-2026_223456_bpb-us-e1.wpmucdn.com.jpeg
🌱 CROP DISEASE DETECTION
-------------------------
Prediction: Potato Late Blight
Confidence: 95.28%


In [ ]:
import numpy as np
import tensorflow as tf

class_names = [
    "Potato Early Blight",
    "Potato Late Blight",
    "Potato Healthy"
]

def predict_disease(image):

    # Resize image
    img = tf.image.resize(image, (160, 160))

    # Add batch dimension
    img = tf.expand_dims(img, axis=0)

    # Predict
    predictions = model.predict(img, verbose=0)[0]

    index = np.argmax(predictions)
    confidence = predictions[index] * 100

    return class_names[index], confidence

In [ ]:
import gradio as gr

def app_predict(image):

    disease, confidence = predict_disease(image)

    if disease == "Potato Early Blight":
        advice = """
Management Guidance:
• Remove severely affected leaves.
• Maintain proper field sanitation.
• Avoid unnecessary leaf wetness.
• Monitor nearby plants regularly.
"""

    elif disease == "Potato Late Blight":
        advice = """
Management Guidance:
• Inspect surrounding plants immediately.
• Remove severely affected plant material.
• Avoid excessive moisture on foliage.
• Follow locally recommended agricultural practices.
"""

    else:
        advice = """
Management Guidance:
• Crop appears healthy.
• Continue regular monitoring.
• Maintain proper irrigation and field sanitation.
"""

    return f"""
🌱 CROP DISEASE DETECTION

Disease/Condition: {disease}

Confidence: {confidence:.2f}%

{advice}
"""


demo = gr.Interface(
    fn=app_predict,
    inputs=gr.Image(
        type="numpy",
        label="Upload Potato Leaf Image"
    ),
    outputs=gr.Textbox(
        label="AI Analysis"
    ),
    title="🌾 AI-Based Crop Disease Detection",
    description="Upload a potato leaf image for AI-based disease detection and management guidance."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8a8f7114f2426b9e90.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
model.save("potato_leaf_model.h5")

In [ ]:
import os

print(os.listdir())

['.config', 'potato_leaf_model.h5', 'potato_dataset', 'Screenshot_18-9-2026_223456_bpb-us-e1.wpmucdn.com.jpeg', 'PlantVillage-Dataset', '.gradio', 'sample_data']


In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.2 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

# -----------------------------
# Page Configuration
# -----------------------------
st.set_page_config(
    page_title="Potato Leaf Disease Detection",
    page_icon="🌿",
    layout="centered"
)

# -----------------------------
# Load Model
# -----------------------------
model = tf.keras.models.load_model("potato_leaf_model.h5")

# Class names
class_names = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

# -----------------------------
# Header
# -----------------------------
st.title("🌿 Potato Leaf Disease Detection")
st.subheader("AI-powered plant disease identification")

st.write(
    "Upload a potato leaf image and our machine learning model "
    "will analyze it and identify whether the leaf is Healthy, "
    "affected by Early Blight, or affected by Late Blight."
)

st.divider()

# -----------------------------
# Image Upload
# -----------------------------
uploaded_file = st.file_uploader(
    "📷 Upload a Potato Leaf Image",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    image = Image.open(uploaded_file).convert("RGB")

    st.image(
        image,
        caption="Uploaded Leaf",
        use_container_width=True
    )

    st.write("### 🔍 Analysis")

    if st.button("🚀 Predict Disease"):

        # Resize image
        img = image.resize((224, 224))

        # Convert to array
        img_array = np.array(img)

        # Normalize
        img_array = img_array / 255.0

        # Add batch dimension
        img_array = np.expand_dims(img_array, axis=0)

        # Prediction
        prediction = model.predict(img_array)

        predicted_class = np.argmax(prediction[0])
        confidence = np.max(prediction[0]) * 100

        disease = class_names[predicted_class]

        st.divider()

        # -----------------------------
        # Result
        # -----------------------------
        st.success(f"🌱 Result: {disease}")

        st.metric(
            "Confidence",
            f"{confidence:.2f}%"
        )

        # -----------------------------
        # Information
        # -----------------------------
        if disease == "Healthy":

            st.info(
                "✅ The potato leaf appears healthy. "
                "Continue regular monitoring and proper plant care."
            )

        elif disease == "Early Blight":

            st.warning(
                "⚠️ Early Blight detected. "
                "Monitor the affected plant and consider appropriate "
                "crop protection measures."
            )

        elif disease == "Late Blight":

            st.error(
                "🚨 Late Blight detected. "
                "The plant should be inspected carefully and "
                "appropriate disease-management measures considered."
            )

st.divider()

st.caption(
    "AI-based Potato Leaf Disease Detection • SIH Project Prototype"
)

Writing app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt & echo $!

4382


In [ ]:
!cat /content/logs.txt



2026-09-19 00:48:24.017 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.168.216.152:8501



In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠙

In [ ]:
!lt --port 8501

your url is: https://chilly-bugs-sneeze.loca.lt
^C


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# Load trained model
model = tf.keras.models.load_model("potato_leaf_model.h5")

class_names = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

def predict_disease(image):

    if image is None:
        return "Please upload a potato leaf image."

    # Resize according to model input
    image = image.resize((160, 160))

    # Convert image to array
    img_array = np.array(image)

    # Normalize pixel values
    img_array = img_array / 255.0

    # Add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # Make prediction
    prediction = model.predict(img_array, verbose=0)

    # Get predicted class
    predicted_class = np.argmax(prediction[0])

    # Get confidence
    confidence = np.max(prediction[0]) * 100

    disease = class_names[predicted_class]

    return f"{disease} — {confidence:.2f}% confidence"


demo = gr.Interface(
    fn=predict_disease,
    inputs=gr.Image(
        type="pil",
        label="Upload Potato Leaf"
    ),
    outputs=gr.Textbox(
        label="Prediction"
    ),
    title="🌿 Potato Leaf Disease Detection",
    description="Upload a potato leaf image to detect Early Blight, Late Blight, or Healthy."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6e22a8bd535231127.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)

Model input shape: (None, 160, 160, 3)
Model output shape: (None, 3)


In [ ]:
import os

print(os.listdir("potato_dataset"))

['Potato___healthy', 'Potato___Early_blight', 'Potato___Late_blight']


In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np

# Load model
model = tf.keras.models.load_model("potato_leaf_model.h5")

class_names = [
    "Early Blight",
    "Late Blight",
    "Healthy"
]

disease_info = {
    "Early Blight": {
        "symptoms": "Dark brown spots with concentric rings on older leaves.",
        "action": "Remove severely affected leaves and monitor the plant regularly.",
        "prevention": "Maintain proper spacing, avoid excess moisture on leaves, and remove infected plant material."
    },

    "Late Blight": {
        "symptoms": "Dark water-soaked spots that can rapidly expand across the leaf.",
        "action": "Isolate affected plants and seek appropriate crop-protection guidance.",
        "prevention": "Avoid prolonged leaf wetness, improve air circulation, and monitor plants frequently."
    },

    "Healthy": {
        "symptoms": "No significant disease symptoms detected.",
        "action": "Continue regular monitoring and maintain proper plant care.",
        "prevention": "Use healthy planting material and maintain good field hygiene."
    }
}


def predict_disease(image):

    if image is None:
        return "Please upload an image.", "", "", ""

    # Resize to model input size
    image = image.resize((160, 160))

    # IMPORTANT:
    # Do NOT divide by 255.
    # The model already contains Rescaling(1./127.5, offset=-1)
    img_array = np.array(image).astype("float32")

    # Add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # Prediction
    prediction = model.predict(img_array, verbose=0)[0]

    predicted_class = np.argmax(prediction)
    confidence = prediction[predicted_class] * 100

    disease = class_names[predicted_class]

    info = disease_info[disease]

    result = f"🌱 {disease}\n\n📊 Confidence: {confidence:.2f}%"

    return (
        result,
        f"🔎 Symptoms\n\n{info['symptoms']}",
        f"💡 Recommended Action\n\n{info['action']}",
        f"🛡️ Prevention\n\n{info['prevention']}"
    )


with gr.Blocks(title="Potato Leaf Disease Detection") as demo:

    gr.Markdown("""
    # 🌿 Potato Leaf Disease Detection
    ### AI-powered potato crop disease identification

    Upload a potato leaf image to detect **Early Blight, Late Blight, or Healthy**.
    """)

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                type="pil",
                label="📷 Upload Potato Leaf"
            )

            predict_button = gr.Button(
                "🔍 Analyze Leaf",
                variant="primary"
            )

        with gr.Column():

            result = gr.Textbox(
                label="🌱 Prediction",
                lines=3
            )

            symptoms = gr.Textbox(
                label="🔎 Symptoms",
                lines=4
            )

    with gr.Row():

        action = gr.Textbox(
            label="💡 Recommended Action",
            lines=4
        )

        prevention = gr.Textbox(
            label="🛡️ Prevention",
            lines=4
        )

    predict_button.click(
        predict_disease,
        inputs=image_input,
        outputs=[
            result,
            symptoms,
            action,
            prevention
        ]
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab825c2087ce6b41e7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
